In [ ]:
import uuid
import os
from typing import Literal, TypedDict, Optional, Dict
from IPython.display import Image, display
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()


print(os.getenv("AZURE_OPENAI_ENDPOINT"))

In [ ]:
# Pydantic model that defines properties of a bicep template for azure resource deployment
class parsed_input(BaseModel):
     """Structured model for Azure resource deployment parameters."""
     
     # Core required fields
     resource_type: str = Field(..., description="Type of Azure resource (storage_account, key_vault, app_service_plan, application_insights, function_app)")
     region: str = Field(..., description="Azure region (e.g., East US, West Europe)")
     name: str = Field(..., description="Name for the resource")
     resource_group: str = Field(default="rg-deployment-test", description="Resource group name")
     
     # Common fields across resources
     sku: Optional[str] = Field(None, description="SKU name (e.g., Standard_LRS, Standard, B1)")
     tier: Optional[str] = Field(None, description="Pricing tier (Basic, Standard, Premium)")
     kind: Optional[str] = Field(None, description="Resource kind (e.g., StorageV2, BlobStorage)")
     api_version: Optional[str] = Field(None, description="Specific API version to use")
     
     # Capacity and scaling
     capacity: Optional[int] = Field(None, description="Capacity or instance count")
     zone_redundancy_enabled: Optional[bool] = Field(None, description="Enable zone redundancy")
     
     # Storage Account specific
     performance: Optional[str] = Field(None, description="Standard or Premium")
     replication_type: Optional[str] = Field(None, description="LRS, GRS, RAGRS, ZRS, GZRS, RAGZRS")
     access_tier: Optional[str] = Field(None, description="Hot, Cool, or Archive")
     enable_https_traffic_only: Optional[bool] = Field(None, description="Enforce HTTPS traffic only")
     min_tls_version: Optional[str] = Field(None, description="TLS1_0, TLS1_1, or TLS1_2")
     allow_blob_public_access: Optional[bool] = Field(None, description="Allow blob public access")
     
     # Key Vault specific
     soft_delete_enabled: Optional[bool] = Field(None, description="Enable soft delete")
     soft_delete_retention_days: Optional[int] = Field(None, description="Retention days for soft delete (7-90)")
     purge_protection_enabled: Optional[bool] = Field(None, description="Enable purge protection")
     enable_rbac_authorization: Optional[bool] = Field(None, description="Enable RBAC authorization")
     
     # App Service Plan specific
     reserved: Optional[bool] = Field(None, description="True for Linux, False for Windows")
     per_site_scaling: Optional[bool] = Field(None, description="Enable per-site scaling")
     
     # Application Insights specific
     application_type: Optional[str] = Field(None, description="Application type (web, other)")
     workspace_resource_id: Optional[str] = Field(None, description="Log Analytics workspace resource ID")
     
     # Function App specific
     runtime: Optional[str] = Field(None, description="Runtime stack (dotnet, node, python, java)")
     runtime_version: Optional[str] = Field(None, description="Runtime version")
     operating_system: Optional[str] = Field(None, description="Operating system (Windows, Linux)")
     app_service_plan_id: Optional[str] = Field(None, description="App Service Plan resource ID")
     storage_account_name: Optional[str] = Field(None, description="Storage account name for function app")
     enable_managed_identity: Optional[bool] = Field(None, description="Enable system-assigned managed identity")
     always_on: Optional[bool] = Field(None, description="Enable always-on feature")
     
     # Networking
     public_network_access: Optional[str] = Field(None, description="Enabled or Disabled")
     
     # Tagging
     tags: Optional[Dict[str, str]] = Field(None, description="Resource tags as key-value pairs")

         


     

In [ ]:
class DeploymentAgentState(TypedDict):
        input_parsed_json: parsed_input
        user_input: str
        infra_code: str
        deployment_status: Literal["Success", "Failure"]
        infra_build_status: Literal["Pass", "Fail"]
        deploy_infra_validate_status: Literal["Pass", "Fail"]
        

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.types import Command, interrupt
from langgraph.graph import END, START, StateGraph
import subprocess
import json
import tempfile
import os
import platform

llm = ChatOpenAI(
    model="gpt-5-mini",  # Fixed model name
    base_url=os.getenv("MICROSOFT_FOUNDRY_ENDPOINT"),  # Use environment variable for base URL
    api_key=os.getenv("MICROSOFT_FOUNDRY_KEY")  # Use environment variable for API key
)

def run_az_command(cmd_args, capture_output=True, text=True, timeout=30):
    """Helper function to run Azure CLI commands with proper Windows support."""
    if platform.system() == "Windows":
        # On Windows, use shell=True to properly handle .cmd files
        cmd = ["az"] + cmd_args
        return subprocess.run(cmd, capture_output=capture_output, text=text, 
                            shell=True, timeout=timeout)
    else:
        # On Unix systems, use the standard approach
        cmd = ["az"] + cmd_args
        return subprocess.run(cmd, capture_output=capture_output, text=text, timeout=timeout)



In [ ]:
def parse_user_input(state: DeploymentAgentState) -> DeploymentAgentState:
    """Parse user input into structured Pydantic model using LLM."""
    parse_prompt = f"""
    You are an expert in parsing user requests for Azure resource deployments.
    Analyze the user request and extract all relevant information to populate the fields.
    
    User Request: {state['user_input']}
    
    Extract the following information and map to the appropriate fields:
    
    REQUIRED FIELDS - Always extract these:
    - resource_type: Identify the Azure resource type from the request
      * For storage: "storage_account"
      * For key vault: "key_vault" 
      * For app service plan: "app_service_plan"
      * For application insights: "application_insights"
      * For function app: "function_app"
    - region: Azure region mentioned (e.g., "East US", "West Europe")
    - name: The specific name provided for the resource
    - resource_group: Resource group name (default: "rg-deployment-test" if not mentioned)
    
    OPTIONAL FIELDS - Extract if mentioned or implied:
    
    For STORAGE ACCOUNT:
    - sku: (e.g., "Standard_LRS", "Premium_LRS")
    - kind: (e.g., "StorageV2", "BlobStorage", "FileStorage")
    - performance: "Standard" or "Premium"
    - replication_type: "LRS", "GRS", "RAGRS", "ZRS", "GZRS", "RAGZRS"
    - access_tier: "Hot", "Cool", or "Archive"
    - enable_https_traffic_only: true/false
    - min_tls_version: "TLS1_2" (recommended)
    - allow_blob_public_access: true/false
    
    For KEY VAULT:
    - sku: "standard" or "premium"
    - tier: "Standard" or "Premium"
    - soft_delete_enabled: true/false
    - soft_delete_retention_days: 7-90 days
    - purge_protection_enabled: true/false
    - enable_rbac_authorization: true/false
    
    For APP SERVICE PLAN:
    - sku: (e.g., "B1", "S1", "P1V2", "Y1" for consumption)
    - tier: (e.g., "Basic", "Standard", "Premium", "Dynamic")
    - reserved: true for Linux, false for Windows
    - capacity: number of instances
    - zone_redundancy_enabled: true/false
    
    For APPLICATION INSIGHTS:
    - application_type: "web" or "other"
    - kind: "web" (typical)
    - workspace_resource_id: Log Analytics workspace ID if specified
    
    For FUNCTION APP:
    - runtime: "dotnet", "node", "python", "java", "powershell"
    - runtime_version: specific version (e.g., "3.9", "14", "6")
    - operating_system: "Windows" or "Linux"
    - app_service_plan_id: if linking to existing plan
    - storage_account_name: required for function apps
    - enable_managed_identity: true/false for system-assigned identity
    - always_on: true/false (not available on Consumption plan)
    
    GENERAL FIELDS:
    - public_network_access: "Enabled" or "Disabled"
    - tags: key-value pairs for resource tagging
    
    IMPORTANT: Only include fields that are explicitly mentioned or can be reasonably inferred from the request.
    Use appropriate defaults where necessary (e.g., resource_group defaults to "rg-deployment-test").
    """
    
    # Use structured output with Pydantic model
    structured_llm = llm.with_structured_output(parsed_input)
    parsed_data = structured_llm.invoke(parse_prompt)
    
    # Convert Pydantic model to dict and store in state
    state['input_parsed_json'] = parsed_data.model_dump(exclude_none=True)
    print("Parsed Input (Pydantic Model):\n", json.dumps(state['input_parsed_json'], indent=2))
    return state


In [ ]:
def parse_user_input_manual(state: DeploymentAgentState) -> DeploymentAgentState:
    """Parse user input into structured JSON using LLM without Pydantic structured output."""
    parse_prompt = f"""
    You are an expert in parsing user requests into structured JSON format for Azure resource deployments.
    Convert the following user request into a structured JSON format:
    User Request: {state['user_input']}
    
    Return a JSON with these fields (include only fields mentioned or implied in the request):
    
    REQUIRED FIELDS:
    - resource_type: the type of Azure resource (e.g., "storage_account", "key_vault", "app_service_plan", "application_insights", "function_app")
    - region: the Azure region (e.g., "East US", "West Europe")
    - name: the name for the resource
    - resource_group: the resource group name (if not specified, use "rg-deployment-test" as default)
    
    COMMON OPTIONAL FIELDS:
    - sku: SKU name (e.g., "Standard_LRS", "Standard", "B1")
    - tier: pricing tier (e.g., "Basic", "Standard", "Premium")
    - kind: resource kind (e.g., "StorageV2", "BlobStorage")
    - api_version: specific API version to use
    - capacity: capacity/instance count
    - zone_redundancy_enabled: true/false for zone redundancy
    
    STORAGE ACCOUNT SPECIFIC:
    - performance: "Standard" or "Premium"
    - replication_type: "LRS", "GRS", "RAGRS", "ZRS", "GZRS", or "RAGZRS"
    - access_tier: "Hot", "Cool", or "Archive"
    - enable_https_traffic_only: true/false (default: true)
    - min_tls_version: "TLS1_0", "TLS1_1", or "TLS1_2"
    - allow_blob_public_access: true/false
    
    KEY VAULT SPECIFIC:
    - soft_delete_enabled: true/false
    - soft_delete_retention_days: number (7-90)
    - purge_protection_enabled: true/false
    - enable_rbac_authorization: true/false
    
    APP SERVICE PLAN SPECIFIC:
    - sku: (e.g., "B1", "S1", "P1V2", "Y1" for consumption)
    - tier: (e.g., "Basic", "Standard", "Premium", "Dynamic")
    - reserved: true for Linux, false for Windows
    - per_site_scaling: true/false
    
    APPLICATION INSIGHTS SPECIFIC:
    - application_type: "web" or "other"
    - workspace_resource_id: Log Analytics workspace ID if specified
    
    FUNCTION APP SPECIFIC:
    - runtime: "dotnet", "node", "python", "java", "powershell"
    - runtime_version: specific version
    - operating_system: "Windows" or "Linux"
    - app_service_plan_id: if linking to existing plan
    - storage_account_name: required for function apps
    - enable_managed_identity: true/false for system-assigned identity
    - always_on: true/false
    
    NETWORKING:
    - public_network_access: "Enabled" or "Disabled"
    
    TAGGING:
    - tags: object with key-value pairs
    
    Return ONLY valid JSON without any markdown formatting or explanations.
    """
    
    # Use simple LLM invoke instead of structured output
    parsed_json = llm.invoke(parse_prompt)
    content = parsed_json.content

    print("parsed content is - " + content)
    try:
        # Remove markdown code blocks if present
        content = content.strip()
        if content.startswith('```'):
            # Remove markdown code fence
            lines = content.split('\n')
            # Remove first line (```json or ```)
            lines = lines[1:]
            # Remove last line if it's ```
            if lines and lines[-1].strip() == '```':
                lines = lines[:-1]
            content = '\n'.join(lines)
        
        # Find the JSON object by looking for the first { and last }
        import re
        first_brace = content.find('{')
        last_brace = content.rfind('}')
        
        if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
            json_str = content[first_brace:last_brace+1]
            parsed_data = json.loads(json_str)
            print("✅ Successfully parsed JSON from LLM response")
        else:
            raise ValueError("No valid JSON object found in response")
            
    except Exception as e:
        print(f"❌ Failed to parse LLM response: {e}")
        print(f"Response content: {content[:200]}...")
        # Fallback parsing
        parsed_data = {
            "resource_type": "storage_account",
            "region": "East US", 
            "name": "mystorageaccount",
            "resource_group": "rg-deployment-test"
        }

    # Ensure resource_group has a default if not parsed
    if 'resource_group' not in parsed_data or not parsed_data['resource_group']:
        parsed_data['resource_group'] = "rg-deployment-test"

    state['input_parsed_json'] = parsed_data
    print("Parsed Input JSON:\n", json.dumps(parsed_data, indent=2))
    return state

In [ ]:
def build_bicep(state: DeploymentAgentState) -> DeploymentAgentState:
    """Build Bicep file before deployment."""
    print("Building Bicep file...")
    
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.bicep', delete=False) as f:
            f.write(state['infra_code'])
            bicep_file = f.name
        
        # Build using Azure CLI with the helper function
        result = run_az_command(["bicep", "build", "--file", bicep_file, "--stdout"])
        
        os.unlink(bicep_file)
        
        if result.returncode == 0:
            print("✅ Bicep build succeeded")
            state['infra_build_status'] = "Pass"
        else:
            print("❌ Bicep build failed:")
            print(result.stderr)
            state['infra_build_status'] = "Fail"
            
    except Exception as e:
        print(f"Error during build: {e}")
        state['infra_build_status'] = "Fail"
    
    return state

In [ ]:
# Add a human review before generating the bicep code
def human_review(state: DeploymentAgentState) -> Command[Literal["deploy_infra_with_cli", END]]:
    """Pause for human review using interrupt and route based on decision"""


    # Interrupt() must come first - any code before it will re-run on resume
    human_decision = interrupt({
        "generated_infra_code": state['infra_code'],
        "action": "Please review and approve/edit this response"
    })

    # Now process the human's decision
    if human_decision.get("approved"):
         return Command(
            update = None,
            goto = "deploy_infra_with_cli"
        )
    else:

        print("Deployment rejected. Ending process.")
        # Rejection means human will handle directly
        return Command(
             update= None,
             goto=END
        ) 

In [ ]:
def generate_infra_code(state: DeploymentAgentState) -> DeploymentAgentState:
    """Generate infrastructure code in bicep based on parsed input using LLM"""
    print("Generating Infrastructure Code...", state['input_parsed_json'])
    
    infra_prompt = f"""
    You are an expert Azure Bicep template generator. Generate a syntactically correct Bicep template based on these requirements:
    {json.dumps(state['input_parsed_json'], indent=2)}
    
    CRITICAL Bicep Syntax Rules - Follow Exactly:
    1. Use ONLY single quotes (') for strings, NEVER double quotes (")
    2. Do NOT use dollar signs ($) anywhere - they are invalid in Bicep
    3. Use camelCase for parameter and variable names
    4. Resource syntax: resourceName 'Microsoft.ResourceProvider/resourceType@apiVersion' = {{}}
    5. Parameter syntax: param parameterName string = 'defaultValue'
    6. Variable syntax: var variableName = 'value'
    7. String interpolation: '${{parameterName}}-suffix' (use ${{}} for interpolation only)
    8. Scope resources to resource group level only (no subscriptions or management groups)
    
    Required Template Structure:
    1. Parameters section (with @description decorators)
    2. Variables section (if needed)
    3. Resources section with ALL specified properties
    4. Outputs section (optional)
    
    RESOURCE-SPECIFIC REQUIREMENTS:
    
    For Storage Account (resource_type: storage_account):
    - API version: '2023-01-01' or higher
    - Resource type: 'Microsoft.Storage/storageAccounts@2023-01-01'
    - Required properties: name, location, sku, kind
    - SKU: Use the parsed sku or default to 'Standard_LRS'
    - Kind: Use parsed kind or default to 'StorageV2'
    - Optional properties: enableHttpsTrafficOnly, minimumTlsVersion, allowBlobPublicAccess, accessTier
    - Name: globally unique, lowercase, 3-24 characters, alphanumeric only
    
    For Key Vault (resource_type: key_vault):
    - API version: '2023-07-01' or higher
    - Resource type: 'Microsoft.KeyVault/vaults@2023-07-01'
    - Required properties: name, location, properties (tenantId, sku, accessPolicies)
    - SKU: Use parsed tier or default to 'standard'
    - Optional: enableSoftDelete, softDeleteRetentionInDays, enablePurgeProtection, enableRbacAuthorization
    - Use tenantId: tenant().tenantId
    
    For App Service Plan (resource_type: app_service_plan):
    - API version: '2023-01-01' or higher
    - Resource type: 'Microsoft.Web/serverfarms@2023-01-01'
    - Required properties: name, location, sku
    - SKU: Must include name (e.g., 'B1', 'S1', 'P1V2', 'Y1') and optionally tier, capacity
    - Optional: reserved (Linux vs Windows), perSiteScaling, zoneRedundant
    
    For Application Insights (resource_type: application_insights):
    - API version: '2020-02-02' or higher
    - Resource type: 'Microsoft.Insights/components@2020-02-02'
    - Required properties: name, location, kind, properties (Application_Type)
    - Kind: Use parsed kind or default to 'web'
    - Application_Type: Use parsed application_type or default to 'web'
    - Optional: WorkspaceResourceId (link to Log Analytics workspace)
    
    For Function App (resource_type: function_app):
    - API version: '2023-01-01' or higher
    - Resource type: 'Microsoft.Web/sites@2023-01-01'
    - Required properties: name, location, kind, properties
    - Kind: 'functionapp' (Windows) or 'functionapp,linux' (Linux)
    - Required in properties: serverFarmId, siteConfig
    - siteConfig should include: appSettings (AzureWebJobsStorage, FUNCTIONS_EXTENSION_VERSION, FUNCTIONS_WORKER_RUNTIME)
    - If enable_managed_identity is true, add identity section with type: 'SystemAssigned'
    - Always create a storage account resource alongside the function app
    - For consumption plan, use serverFarmId with a Dynamic (Y1) App Service Plan
    
    CRITICAL - STORAGE CONNECTION STRING FOR FUNCTION APPS:
    - NEVER use placeholder values like 'REPLACE_WITH_KEY' in connection strings
    - ALWAYS use listKeys() to dynamically retrieve the storage account key at deployment time
    - Build the connection string as a Bicep variable using string interpolation:
      var storageConnectionString = 'DefaultEndpointsProtocol=https;AccountName=${{storageAccount.name}};AccountKey=${{storageAccount.listKeys().keys[0].value}};EndpointSuffix=core.windows.net'
    - Use this variable for BOTH AzureWebJobsStorage and WEBSITE_CONTENTAZUREFILECONNECTIONSTRING app settings
    - For WEBSITE_CONTENTSHARE, use a simple lowercase string derived from the function app name
    
    IMPORTANT: 
    - Include ALL properties specified in the parsed input
    - Use appropriate default values for properties not specified
    - Ensure SKU, tier, capacity, and other configuration properties are correctly applied
    - Add tags if specified in the input
    - Set zone redundancy if specified
    - Configure networking settings (publicNetworkAccess) if specified
    - For function apps, ensure all required app settings are configured
    - NEVER use hardcoded or placeholder secrets/keys in templates - always use dynamic references like listKeys()
    

    Return ONLY the complete, valid Bicep code without markdown formatting or explanations.

    """    

    response = llm.invoke(infra_prompt)
    state['infra_code'] = response.content
    print("Generated Infrastructure Code:\n", response.content)
    return state

In [ ]:
def refine_infra_code(state: DeploymentAgentState) -> DeploymentAgentState:
    """Refine infrastructure code based on validation results using LLM."""
    print("Refining Infrastructure Code based on validation results...")
    
    refine_prompt = f"""
    The following Bicep code has syntax errors. Please refine it to fix the issues while adhering to Bicep syntax rules.
    
    Current Bicep Code:
    {state['infra_code']}
    
    Ensure the refined code follows these rules:
    1. Use ONLY single quotes (') for strings, NEVER double quotes (")
    2. Do NOT use dollar signs ($) anywhere - they are invalid in Bicep
    3. Use camelCase for parameter and variable names
    4. Resource syntax: resourceName 'Microsoft.ResourceProvider/resourceType@apiVersion' = {{}}
    5. Parameter syntax: param parameterName string = 'defaultValue'
    6. Variable syntax: var variableName = 'value'
    7. String interpolation: '${{parameterName}}-suffix' (use ${{}} for interpolation only)
    8. Use string interpolation for any dynamic values for e.g. storage account connection strings in function app settings.
    9. Remove any unnecessary dependencies or incorrect constructs.
    10. NEVER use placeholder values like 'REPLACE_WITH_KEY' for storage keys or secrets.
        For storage connection strings, ALWAYS use listKeys() to dynamically get the key:
        var storageConnectionString = 'DefaultEndpointsProtocol=https;AccountName=${{storageAccount.name}};AccountKey=${{storageAccount.listKeys().keys[0].value}};EndpointSuffix=core.windows.net'
    11. Use this dynamic connection string for AzureWebJobsStorage and WEBSITE_CONTENTAZUREFILECONNECTIONSTRING.
    
    Return ONLY the complete, valid Bicep code without markdown formatting or explanations.
    """
    response = llm.invoke(refine_prompt)
    state['infra_code'] = response.content
    print("Refined Infrastructure Code:\n", response.content[:200] + "...")
    return state


In [ ]:
# Add a step to validate infra code using az bicep validate
def prevalidate_infra_code(state: DeploymentAgentState) -> DeploymentAgentState:
    """Pre-validate Bicep code using Azure CLI before deployment."""
    print("Pre-validating Bicep code...")
    
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.bicep', delete=False) as f:
            f.write(state['infra_code'])
            bicep_file = f.name
        
          # validate the deployment using az deployment grqoup validate
        resource_group = state['input_parsed_json'].get('resource_group', 'rg-aidemo')
        result = run_az_command([
            "deployment", "group", "validate",
            "--resource-group", resource_group, 
            "--template-file", bicep_file
        ])

        
        os.unlink(bicep_file)
        
        if result.returncode == 0:
            print("✅ deployment pre-validation succeeded")
            state['deploy_infra_validate_status'] = "Pass"
        else:
            print("❌ deployment pre-validation failed:")
            print(result.stderr)
            state['deploy_infra_validate_status'] = "Fail"
            
    except Exception as e:
        print(f"Error during deployment pre-validation: {e}")
        state['deploy_infra_validate_status'] = "Fail"
    
    return state

In [ ]:
# this conditional edge will check build_bicep result. 
def conditional_edge_build(state: DeploymentAgentState) -> Literal["prevalidate_infra_code", "refine_infra_code", END] :
    infra_build_status = state["infra_build_status"]
    if infra_build_status == "Pass":
        print("Build passed, proceeding to pre-validate infrastructure code.")
        return "prevalidate_infra_code"
    elif infra_build_status == "Fail":
        print("Build failed, refining infrastructure code.")
        return "refine_infra_code"
    else:
        return END

In [ ]:
# this condition will check prevalidate_infra_code result.
def conditional_edge_prevalidate(state: DeploymentAgentState) -> Literal["human_review", END] :
    deploy_infra_validate_status = state["deploy_infra_validate_status"]
    if deploy_infra_validate_status == "Pass":
        print("Validation passed, proceeding to human review.")
        print("Needs approval")
        return "human_review"
    elif deploy_infra_validate_status == "Fail":
        print("Validation failed, Ending.")
        return END

In [ ]:
def deploy_infra_with_cli(state: DeploymentAgentState) -> DeploymentAgentState:
    """Deploy infrastructure using Azure CLI."""
    print("Deploying Infrastructure using Azure CLI...")
    
    try:
        # Save Bicep code to temporary file
        with tempfile.NamedTemporaryFile(mode='w', suffix='.bicep', delete=False) as f:
            f.write(state['infra_code'])
            bicep_file = f.name
        
        # Use configurable resource group name from parsed input
        resource_group = state['input_parsed_json']['resource_group']
        deployment_name = f"deployment-{state['input_parsed_json']['name']}"
        
        print(f"Using resource group: {resource_group}")
        
        # Check if logged in to Azure using the helper function
        login_check = run_az_command(["account", "show"])
        
        if login_check.returncode != 0:
            print("Not logged in to Azure. Please run 'az login' first.")
            print(f"Error: {login_check.stderr}")
            #state['infra_build_status'] = "Fail"
            return state
        
        # Create resource group if it doesn't exist
        rg_create = run_az_command([
            "group", "create",
            "--name", resource_group,
            "--location", state['input_parsed_json']['region']
        ])
        
        print(f"Resource group creation result: {rg_create.returncode}")
        if rg_create.stderr:
            print(f"RG creation stderr: {rg_create.stderr}")
        
        # Deploy the Bicep template with extended timeout for deployment operations
        print("Starting deployment... This may take several minutes.")
        deploy_result = run_az_command([
            "deployment", "group", "create",
            "--resource-group", resource_group,
            "--template-file", bicep_file,
            "--name", deployment_name,
            "--mode", "Incremental"
        ], timeout=600)  # 10 minutes timeout for deployment
        
        # Clean up temp file
        os.unlink(bicep_file)
        
        if deploy_result.returncode == 0:
            print("✅ Infrastructure deployed successfully!")
            state['deployment_status'] = "Pass"
            print("Deployment output:", deploy_result.stdout[:500] + "...")
        else:
            print("❌ Infrastructure deployment failed!")
            print("Error:", deploy_result.stderr)
            state['deployment_status'] = "Fail"
            
    except subprocess.TimeoutExpired:
        print("❌ Deployment timed out after 10 minutes.")
        print("💡 The deployment might still be running in Azure. Check the Azure portal.")
    except Exception as e:
        print(f"Error during deployment: {e}")
    
    return state

In [ ]:
def verify_deployment(state: DeploymentAgentState) -> DeploymentAgentState:
    """Verify the deployment status using Azure CLI."""
    print("Verifying deployment status...")
    
    try:
        resource_group = state['input_parsed_json']['resource_group']
        deployment_name = f"deployment-{state['input_parsed_json']['name']}"
        
        # Get deployment details using az deployment show
        result = run_az_command([
            "deployment", "group", "show",
            "--resource-group", resource_group,
            "--name", deployment_name,
            "--query", "{provisioningState:properties.provisioningState, timestamp:properties.timestamp, duration:properties.duration, outputs:properties.outputs}",
            "--output", "json"
        ])
        
        if result.returncode == 0:
            deployment_info = json.loads(result.stdout)
            provisioning_state = deployment_info.get('provisioningState', 'Unknown')
            
            print(f"\n{'='*50}")
            print(f"Deployment Verification Results:")
            print(f"{'='*50}")
            print(f"Deployment Name: {deployment_name}")
            print(f"Resource Group: {resource_group}")
            print(f"Provisioning State: {provisioning_state}")
            print(f"Timestamp: {deployment_info.get('timestamp', 'N/A')}")
            print(f"Duration: {deployment_info.get('duration', 'N/A')}")
            
            if deployment_info.get('outputs'):
                print(f"\nOutputs:")
                for key, value in deployment_info['outputs'].items():
                    print(f"  {key}: {value.get('value', 'N/A')}")
            
            print(f"{'='*50}\n")
            
            if provisioning_state == "Succeeded":
                print("✅ Deployment verification: SUCCESS")
                #state['deployment_status'] = "Success"
            else:
                print(f"⚠️ Deployment verification: {provisioning_state}")
                #state['deployment_status'] = "Failure" if provisioning_state == "Failed" else "Success"
        else:
            print("❌ Failed to verify deployment")
            print(f"Error: {result.stderr}")
            #state['deployment_status'] = "Failure"
            
    except Exception as e:
        print(f"Error during deployment verification: {e}")
        #state['deployment_status'] = "Failure"
    
    return {}

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
#config = {"configurable": {"thread_id": "3"}}

In [ ]:
# Define the enhanced state graph
graph = StateGraph(DeploymentAgentState)

# Add all nodes
graph.add_node("parse_user_input", parse_user_input)
graph.add_node("generate_infra_code", generate_infra_code)
graph.add_node("build_bicep", build_bicep)
graph.add_node("refine_infra_code", refine_infra_code)
graph.add_node("prevalidate_infra_code", prevalidate_infra_code)
graph.add_node("human_review", human_review)
graph.add_node("deploy_infra_with_cli", deploy_infra_with_cli)
graph.add_node("verify_deployment", verify_deployment)

# Define the flow
graph.add_edge(START, "parse_user_input")
graph.add_edge("parse_user_input", "generate_infra_code")
graph.add_edge("generate_infra_code", "build_bicep")
graph.add_conditional_edges("build_bicep", conditional_edge_build)
graph.add_edge("refine_infra_code", "build_bicep")
graph.add_conditional_edges("prevalidate_infra_code", conditional_edge_prevalidate)
graph.add_edge("deploy_infra_with_cli", "verify_deployment")
graph.add_edge("verify_deployment", END)


# Compile the graph
app = graph.compile(checkpointer=memory)




In [ ]:
# Option 1: Print ASCII representation (works offline, no dependencies)
print(app.get_graph().print_ascii())

# Option 2: Get Mermaid syntax to copy/paste into https://mermaid.live
print("\n" + "="*50)
print("Copy the text below and paste it into https://mermaid.live to visualize:")
print("="*50)
print(app.get_graph().draw_mermaid())
print("="*50)

In [ ]:



initial_state: DeploymentAgentState = {
   #"user_input": "Create a ADLS storage account named teststorageaccount16120 in resource group rg-aidemo in the East US region with standard performance and locally-redundant storage"
   #"user_input": "create a key vault named kv-aidemo-140126 in resource group rg-aidemo in the East US region with standard tier and soft delete enabled"
   #"user_input": "create an AppService Plan named appserviceplan-aidemo-290326 in resource group rg-aidemo in the East US region with B1 pricing tier. Disable Zone redundancy."
   #"user_input" : "create an application insights resource named ai-aidemo-290326 in resource group rg-aidemo in the East US region"
   "user_input": "create a function app named funcapp-aidemo-10022026 in resource group rg-aidemo in the East US region with consumption plan and enable system assigned managed identity."
}

# Run with a thread_id for persistence
config = {"configurable": {"thread_id": "5"}}
result = app.invoke(initial_state, config)

#result = app.invoke(initial_state)

In [ ]:
print(result["__interrupt__"])

In [ ]:
# The graph will pause at human_review
print(f"Draft ready for review: {result['infra_code']}\n")



In [ ]:
# Provide human input to resume
human_response = Command(
    resume = {
        "approved": True  # Change to True to approve the deployment
    }
)

# Resume execution
final_result = app.invoke(human_response, config)
print(final_result)

In [ ]:
import gradio as gr
import uuid

# Track active thread and pending interrupt state
active_thread_id = None
pending_interrupt = False
validation_passed = False

def run_workflow(user_prompt):
    """Run the LangGraph workflow from user prompt up to the human_review interrupt."""
    global active_thread_id, pending_interrupt, validation_passed
    
    validation_passed = False
    
    if not user_prompt.strip():
        return "Please enter a deployment request.", "", "", gr.update(interactive=False)
    
    active_thread_id = str(uuid.uuid4())
    thread_config = {"configurable": {"thread_id": active_thread_id}}
    
    initial_state: DeploymentAgentState = {"user_input": user_prompt}
    
    log_lines = []
    
    # Capture print output
    import io, contextlib
    f = io.StringIO()
    with contextlib.redirect_stdout(f):
        result = app.invoke(initial_state, thread_config)
    
    log_lines.append(f.getvalue())
    
    # Check if we hit the human_review interrupt (only reached when validation passed)
    interrupt_info = result.get("__interrupt__")
    infra_code = result.get("infra_code", "No infrastructure code generated.")
    validation_passed = result.get("deploy_infra_validate_status") == "Pass"
    
    if interrupt_info:
        pending_interrupt = True
        #validation_passed = True
        log_lines.append("\n⏸️ Workflow paused at human review. Review the Bicep code and Approve or Reject.")
    else:
        pending_interrupt = False
       # validation_passed = False
        log_lines.append("\n⚠️ Workflow completed without reaching human review (may have failed at an earlier step).")
    
    return "\n".join(log_lines), infra_code, "", gr.update(interactive=validation_passed)

def approve_deployment():
    """Resume workflow with approval."""
    global active_thread_id, pending_interrupt
    
    if not pending_interrupt or not active_thread_id:
        return "No pending deployment to approve.", ""
    
    thread_config = {"configurable": {"thread_id": active_thread_id}}
    human_response = Command(resume={"approved": True})
    
    import io, contextlib
    f = io.StringIO()
    with contextlib.redirect_stdout(f):
        final_result = app.invoke(human_response, thread_config)
    
    pending_interrupt = False
    log_output = f.getvalue()
    
    status = final_result.get("deployment_status", "Unknown")
    log_output += f"\n\n{'='*50}\nFinal Deployment Status: {status}\n{'='*50}"
    
    return log_output, status

def reject_deployment():
    """Resume workflow with rejection."""
    global active_thread_id, pending_interrupt
    
    if not pending_interrupt or not active_thread_id:
        return "No pending deployment to reject.", ""
    
    thread_config = {"configurable": {"thread_id": active_thread_id}}
    human_response = Command(resume={"approved": False})
    
    import io, contextlib
    f = io.StringIO()
    with contextlib.redirect_stdout(f):
        final_result = app.invoke(human_response, thread_config)
    
    pending_interrupt = False
    return f.getvalue() + "\n\n❌ Deployment rejected by user.", "Rejected"

# Build the Gradio UI
with gr.Blocks(title="Azure Deployment Agent", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 Azure Infrastructure Deployment Agent")
    gr.Markdown("Enter a natural language request to deploy Azure resources. The agent will parse your request, generate Bicep code, validate it, and deploy after your approval.")
    
    with gr.Row():
        with gr.Column(scale=2):
            user_input = gr.Textbox(
                label="Deployment Request",
                placeholder="e.g., Create a storage account named mystorageacct01 in resource group rg-aidemo in East US with standard performance and LRS replication",
                lines=3
            )
            submit_btn = gr.Button("🔄 Run Workflow", variant="primary")
        
        with gr.Column(scale=1):
            gr.Markdown("### Example Prompts")
            gr.Markdown("""
- Create a storage account named mysa123 in rg-aidemo in East US with standard LRS
- Create a key vault named kv-demo-01 in rg-aidemo in West Europe with soft delete enabled
- Create an App Service Plan named asp-demo in rg-aidemo in East US with B1 tier
- Create a function app named func-demo in rg-aidemo in East US with consumption plan and managed identity
            """)
    
    with gr.Row():
        workflow_log = gr.Textbox(label="Workflow Log", lines=15, interactive=False)
    
    with gr.Row():
        bicep_code = gr.Code(label="Generated Bicep Code", language=None, lines=20)
    
    with gr.Row():
        
        approve_btn = gr.Button("✅ Approve & Deploy", variant="primary", interactive=False)
        reject_btn = gr.Button("❌ Reject", variant="stop")
    
    with gr.Row():
        deploy_log = gr.Textbox(label="Deployment Log", lines=10, interactive=False)
        deploy_status = gr.Textbox(label="Deployment Status", interactive=False)
    
    # Wire up events
    submit_btn.click(
        fn=run_workflow,
        inputs=[user_input],
        outputs=[workflow_log, bicep_code, deploy_log, approve_btn]
    )
    
    approve_btn.click(
        fn=approve_deployment,
        inputs=[],
        outputs=[deploy_log, deploy_status]
    )
    
    reject_btn.click(
        fn=reject_deployment,
        inputs=[],
        outputs=[deploy_log, deploy_status]
    )

demo.launch(share=False)